# LG HelloDoctor — C팀 RAG 파이프라인 최종
> 크롤링 → 공공API → ChromaDB → Kakao · HIRA 병원검색 → 응급판단 → Tool Router

## Step 1 — 라이브러리 설치

In [ ]:
!pip install chromadb sentence-transformers requests python-dotenv -q
print('설치 완료!')

# ──────────────────────────────────────────────────────────────────────────────
# Google Colab 환경 설정 (로컬에서는 무시됨)
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True
    print('✓ Google Drive 마운트 완료')
except ImportError:
    IN_COLAB = False
    print('ℹ 로컬 환경 감지 (Colab 아님)')

print(f'IN_COLAB: {IN_COLAB}')


설치 완료!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — API 키 설정

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

KAKAO_API_KEY = os.getenv('KAKAO_API_KEY')
HIRA_API_KEY  = os.getenv('DATA_API_KEY')

# ── DB 경로 선택 (Colab/로컬) ──────────────────────────────────────────────────
if IN_COLAB:
    # Colab: Google Drive의 C_Mading 폴더 기준
    DB_PATH = '/content/drive/MyDrive/LG_HelloDoctor/C_Mading/chroma_db'
else:
    # 로컬: 현재 디렉토리 기준
    DB_PATH = os.path.join(os.getcwd(), 'C_Mading', 'chroma_db')

os.makedirs(DB_PATH, exist_ok=True)
print(f'API 키 로드 완료!')
print(f'DB 경로: {DB_PATH}')
print(f'존재 여부: {os.path.exists(DB_PATH)}')

API 키 로드 완료!
DB 경로: c:\Users\juyeon\Desktop\project\LGHelloDoctor\RAG\db


## Step 3 — 국가건강정보포털 크롤링 (질환 + 복약 + 응급)

In [ ]:
import requests
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Referer': 'https://health.kdca.go.kr/healthinfo/biz/health/unifiedSearch/unifiedSearchMain.do'
}
VIEW_URL = 'https://health.kdca.go.kr/healthinfo/biz/health/gnrlzHealthInfo/gnrlzHealthInfo/gnrlzHealthInfoView.do'

# (이름, 카테고리, cntnts_sn) — 2026년 4월 기준
CRAWL_TARGETS = [
    # 질환 정보
    ('무릎관절염',      '증상_진료과', '5969'),
    ('허리디스크',      '증상_진료과', '3348'),
    ('오십견',          '증상_진료과', '1567'),
    ('고혈압',          '증상_진료과', '6765'),
    ('당뇨병',          '증상_진료과', '5305'),
    ('심근경색',        '증상_진료과', '6770'),
    ('뇌졸중',          '증상_진료과', '5495'),
    ('위염',            '증상_진료과', '6777'),
    ('역류성식도염',    '증상_진료과', '2057'),
    ('폐렴',            '증상_진료과', '5249'),
    ('천식',            '증상_진료과', '6784'),
    ('편두통',          '증상_진료과', '6557'),
    ('어지럼증',        '증상_진료과', '6550'),
    ('아토피',          '증상_진료과', '6582'),
    ('두드러기',        '증상_진료과', '6581'),
    # 복약 안내
    ('항생제',          '복약_안내',   '6475'),
    ('노인 약물복용',   '복약_안내',   '5428'),
    ('당뇨환자 식이요법', '복약_안내', '3388'),
    ('당뇨환자 운동요법', '복약_안내', '3390'),
    # 응급 안내
    ('심폐소생술',      '응급_안내',   '6226'),
    ('동물·곤충 응급',  '응급_안내',   '5483'),
    # 추가 질환 (b_output_1000 커버리지 확장)
    ('수면장애',        '증상_진료과', '6558'),
    ('갑상선기능저하증','증상_진료과', '6782'),
    ('갑상선기능항진증','증상_진료과', '6783'),
    ('협심증',          '증상_진료과', '6769'),
    ('소화불량',        '증상_진료과', '6776'),
    ('이명',            '증상_진료과', '5706'),
    ('결막염',          '증상_진료과', '6583'),
    ('치통',            '증상_진료과', '6584'),
    ('사랑니',          '증상_진료과', '6585'),
    ('골절',            '증상_진료과', '5543'),
    ('요로감염',        '증상_진료과', '6586'),
    ('두통',            '증상_진료과', '6553'),
    ('불면증',          '증상_진료과', '6559'),
    ('폐결핵',          '증상_진료과', '5250'),
    ('빈혈',            '증상_진료과', '1104'),
]

DISEASE_DEPT_MAP = {
    '무릎관절염': '정형외과', '허리디스크': '정형외과', '오십견': '정형외과',
    '고혈압': '내과',         '당뇨병': '내과',         '심근경색': '심장내과',
    '뇌졸중': '신경과',       '위염': '소화기내과',     '역류성식도염': '소화기내과',
    '폐렴': '내과',           '천식': '호흡기내과',     '편두통': '신경과',
    '어지럼증': '이비인후과', '아토피': '피부과',       '두드러기': '피부과',
    '수면장애': '신경과',     '갑상선기능저하증': '내과', '갑상선기능항진증': '내과',
    '협심증': '심장내과',     '소화불량': '소화기내과', '이명': '이비인후과',
    '결막염': '안과',         '치통': '치과',            '사랑니': '치과',
    '골절': '정형외과',       '요로감염': '비뇨의학과', '두통': '신경과',
    '불면증': '신경과',       '폐결핵': '내과',          '빈혈': '내과',
}

def crawl_health_info(name: str, cntnts_sn: str) -> dict:
    try:
        response = requests.post(VIEW_URL, headers=HEADERS, data={'cntnts_sn': cntnts_sn}, timeout=10)
        response.encoding = 'utf-8'
        content_parts = []
        for div_id in ['contentsDiv3', 'contentsDiv4', 'contentsDiv5']:
            if f'id="{div_id}"' in response.text:
                idx = response.text.index(f'id="{div_id}"')
                section = response.text[idx:idx+3000]
                clean = re.sub(r'<[^>]+>', ' ', section)
                clean = re.sub(r'\s+', ' ', clean).strip()
                content_parts.append(clean[:400])
        content = ' '.join(content_parts)[:500].strip()
        return {'text': content, 'success': bool(content)}
    except:
        return {'text': '', 'success': False}


ALL_DOCUMENTS = []
print('국가건강정보포털 크롤링 중...')
for name, category, sn in CRAWL_TARGETS:
    result = crawl_health_info(name, sn)
    time.sleep(0.5)
    if result['success']:
        dept = DISEASE_DEPT_MAP.get(name, '')
        text = f"{name}: {result['text']} (진료과: {dept})" if dept else result['text']
        ALL_DOCUMENTS.append({
            'id':       f'crawl_{name}',
            'text':     text,
            'category': category,
            'source':   'health.kdca.go.kr'
        })
        print(f'v {name}')
    else:
        print(f'x {name} 실패')

print(f'크롤링 완료: {len(ALL_DOCUMENTS)}개')


국가건강정보포털 크롤링 중...
v 무릎관절염
v 허리디스크
v 오십견
v 고혈압
v 당뇨병
v 심근경색
v 뇌졸중
v 위염
v 역류성식도염
v 폐렴
v 천식
v 편두통
v 어지럼증
v 아토피
v 두드러기
v 항생제
v 노인 약물복용
v 당뇨환자 식이요법
v 당뇨환자 운동요법
v 심폐소생술
v 동물·곤충 응급
크롤링 완료: 21개


## Step 4 — ChromaDB 인덱스 구축

In [30]:
import chromadb
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 로드 완료!')

chroma_client = chromadb.PersistentClient(path=DB_PATH)

try:
    chroma_client.delete_collection('medical_knowledge')
except:
    pass

collection = chroma_client.create_collection(
    name='medical_knowledge',
    metadata={'hnsw:space': 'cosine'}
)

texts      = [doc['text']     for doc in ALL_DOCUMENTS]
ids        = [doc['id']       for doc in ALL_DOCUMENTS]
metas      = [{'category': doc['category'], 'source': doc['source']} for doc in ALL_DOCUMENTS]
embeddings = embed_model.encode(texts).tolist()

collection.upsert(ids=ids, documents=texts, embeddings=embeddings, metadatas=metas)
print(f'ChromaDB 구축 완료: {collection.count()}개 저장')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18088.69it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


임베딩 모델 로드 완료!
ChromaDB 구축 완료: 21개 저장


## Step 5 — RAG 파이프라인 (Query Rewriting + Hybrid Search + Reranking)

In [31]:
QUERY_REWRITE_MAP = {
    '무릎': '무릎관절염 정형외과 관절 통증 진료',
    '허리': '허리디스크 정형외과 척추 통증 진료',
    '어깨': '오십견 정형외과 어깨 통증 진료',
    '머리': '편두통 신경과 두통 진료',
    '배':   '위염 소화불량 소화기내과 진료',
    '가슴': '심근경색 심장내과 흉통 진료',
    '눈':   '안과 시력 결막염 진료',
    '귀':   '이비인후과 이명 난청 진료',
    '피부': '아토피 피부과 발진 진료',
    '소변': '비뇨의학과 방광 요로 진료',
    '혈압': '고혈압 내과 혈압약 복용',
    '당뇨': '당뇨병 내과 당뇨약 복용',
    '혈압약': '혈압약 복용 방법 주의사항',
    '당뇨약': '당뇨약 복용 방법 주의사항',
    '감기약': '감기약 복용 방법 주의사항',
}

def query_rewrite(query: str) -> str:
    for kw, rewritten in QUERY_REWRITE_MAP.items():
        if kw in query:
            return rewritten
    return query

def vector_search(query: str, n_results: int = 5) -> list:
    rewritten = query_rewrite(query)
    query_emb = embed_model.encode([rewritten]).tolist()
    results   = collection.query(query_embeddings=query_emb, n_results=n_results)
    return [{'text': results['documents'][0][i], 'category': results['metadatas'][0][i]['category'], 'source': results['metadatas'][0][i]['source']} for i in range(len(results['documents'][0]))]

def keyword_search(query: str) -> list:
    keywords = query.split()
    matched  = []
    for doc in ALL_DOCUMENTS:
        score = sum(1 for kw in keywords if kw in doc['text'])
        if score > 0:
            matched.append({'text': doc['text'], 'category': doc['category'], 'source': doc['source'], 'score': score})
    matched.sort(key=lambda x: x['score'], reverse=True)
    return matched[:3]

def rerank(query: str, docs: list) -> list:
    from numpy import dot
    from numpy.linalg import norm
    q_emb    = embed_model.encode([query])
    d_embs   = embed_model.encode([d['text'] for d in docs])
    scores   = [(dot(q_emb[0], d_emb) / (norm(q_emb[0]) * norm(d_emb)), docs[i]) for i, d_emb in enumerate(d_embs)]
    scores.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scores]

def full_rag_pipeline(query: str) -> str:
    vector_results  = vector_search(query, n_results=5)
    keyword_results = keyword_search(query)
    seen = set()
    combined = []
    for r in vector_results + keyword_results:
        if r['text'] not in seen:
            seen.add(r['text'])
            combined.append(r)
    reranked = rerank(query, combined)
    return ' '.join([r['text'] for r in reranked[:3]])

print('RAG 파이프라인 준비 완료!')

RAG 파이프라인 준비 완료!


## Step 6 — 병원 검색 (Kakao + HIRA 통합)

In [32]:
SYMPTOM_DEPT_MAP = {
    '무릎': ('정형외과', '05'), '허리': ('정형외과', '05'),
    '어깨': ('정형외과', '05'), '발목': ('정형외과', '05'),
    '눈':   ('안과', '12'),     '귀':   ('이비인후과', '13'),
    '코':   ('이비인후과', '13'), '목':  ('이비인후과', '13'),
    '치아': ('치과', '21'),     '잇몸': ('치과', '21'),
    '피부': ('피부과', '14'),   '소변': ('비뇨의학과', '15'),
    '머리': ('신경과', '02'),   '가슴': ('심장내과', '01'),
    '배':   ('소화기내과', '01'), '당뇨': ('내과', '01'),
    '혈압': ('내과', '01'),     '기침': ('내과', '01'),
}

DEPT_NAME_TO_CODE = {v[0]: v[1] for v in SYMPTOM_DEPT_MAP.values()}

def search_kakao(dept_name: str, lat: float = 37.5012, lng: float = 127.0396) -> list:
    url     = 'https://dapi.kakao.com/v2/local/search/keyword.json'
    headers = {'Authorization': f'KakaoAK {KAKAO_API_KEY}'}
    params  = {'query': dept_name, 'x': lng, 'y': lat, 'radius': 2000, 'category_group_code': 'HP8', 'size': 5}
    try:
        response = requests.get(url, headers=headers, params=params, timeout=5)
        return [{'name': p['place_name'], 'address': p['road_address_name'] or p['address_name'],
                 'phone': p['phone'], 'distance': int(p['distance']), 'source': 'kakao'}
                for p in response.json().get('documents', [])]
    except:
        return []

def search_hira(dept_code: str, sido: str = '110000') -> list:
    url    = 'https://apis.data.go.kr/B551182/hospInfoServicev2/getHospBasisList'
    params = {'serviceKey': HIRA_API_KEY, 'pageNo': 1, 'numOfRows': 5, 'sidoCd': sido, 'dgsbjtCd': dept_code}
    try:
        response = requests.get(url, params=params, timeout=10)
        root     = ET.fromstring(response.text)
        return [{'name': item.findtext('yadmNm', ''), 'address': item.findtext('addr', ''),
                 'phone': item.findtext('telno', ''), 'source': 'hira'}
                for item in root.findall('.//item')]
    except:
        return []

def search_hospital(b_input: dict, lat: float = 37.5012, lng: float = 127.0396) -> dict:
    entities  = b_input.get('entities', {})
    body_part = entities.get('body_part') or ''
    symptom   = entities.get('symptom') or ''
    true_dept = b_input.get('true_dept')  # B팀이 준 진료과 직접 사용

    # 1) true_dept 있으면 바로 사용
    if true_dept:
        dept_name = true_dept
        dept_code = DEPT_NAME_TO_CODE.get(true_dept, '01')
    else:
        # 2) body_part / symptom -> 진료과 매핑
        dept_name, dept_code = '내과', '01'
        for kw, (name, code) in SYMPTOM_DEPT_MAP.items():
            if kw in body_part or kw in symptom:
                dept_name, dept_code = name, code
                break

    kakao = sorted([h for h in search_kakao(dept_name, lat, lng) if h['phone']], key=lambda x: x['distance'])
    hira  = search_hira(dept_code)
    return {'department': dept_name, 'nearby': kakao[:3], 'official': hira[:3]}

print('병원 검색 준비 완료!')


병원 검색 준비 완료!


## Step 7 — 응급 판단 (심각도 점수화)

In [33]:
EMERGENCY_SCORES = {
    '숨이 안 쉬어': 100, '의식이 없': 100, '심장이 멎': 100,
    '피를 토': 90,       '가슴이 너무 아프': 90, '한쪽이 마비': 90,
    '말이 어눌': 85,     '입이 돌아': 85,
    '갑자기 안 보여': 80, '쓰러': 80, '혈압이 200': 80,
    '약을 잘못': 75,     '뼈가 부러': 70, '화상': 65,
    '식은땀': 30,        '가슴이 아파': 40,
    '어지러': 20,        '두통': 15,
}

def emergency_check(text: str) -> dict:
    total, matched = 0, []
    for kw, score in EMERGENCY_SCORES.items():
        if kw in text:
            total += score
            matched.append(kw)
    if len(matched) >= 2:
        total = min(total * 1.2, 100)
    if total >= 70:
        return {'is_emergency': True,  'severity': 'HIGH',   'score': round(total), 'action': '지금 바로 119에 전화해 주세요.'}
    elif total >= 40:
        return {'is_emergency': True,  'severity': 'MEDIUM', 'score': round(total), 'action': '응급실에 가보시는 게 좋을 것 같아요.'}
    else:
        return {'is_emergency': False, 'severity': 'LOW',    'score': round(total), 'action': None}

print('응급 판단 준비 완료!')

응급 판단 준비 완료!


## Step 8 — Tool Router

In [34]:
def tool_router(input_from_B: dict, lat: float = 37.5012, lng: float = 127.0396) -> dict:
    intent     = input_from_B.get('intent', 'symptom_inquiry')
    confidence = input_from_B.get('confidence', 1.0)
    b_severity = input_from_B.get('severity')  # B팀 severity

    # 멀티턴: turn2_text 있으면 합쳐서 풍부한 쿼리 구성
    turn1 = input_from_B.get('turn1_text') or ''
    turn2 = input_from_B.get('turn2_text') or ''
    query = (turn1 + ' ' + turn2).strip() if turn2 else (input_from_B.get('query') or turn1)

    result = {'intent': intent, 'rag_context': None, 'hospitals': None, 'emergency': None}

    # 1) 응급: B팀 severity 우선, 없으면 자체 판단
    if b_severity == 'HIGH' or intent == 'emergency':
        result['emergency'] = {'is_emergency': True, 'severity': 'HIGH', 'action': '지금 바로 119에 전화해 주세요.'}
        return result

    emerg = emergency_check(query)
    if emerg['is_emergency']:
        result['emergency'] = emerg
        if emerg['severity'] == 'HIGH':
            return result

    # 2) confidence 낮으면 경고 플래그
    low_confidence = confidence < 0.75

    # 3) intent별 처리
    if intent == 'symptom_inquiry':
        result['rag_context'] = full_rag_pipeline(query)
        result['hospitals']   = search_hospital(input_from_B, lat, lng)  # entities + true_dept 활용
        if low_confidence:
            result['rag_context'] += ' (정확하지 않을 수 있습니다. 증상을 다시 말씀해 주세요.)'

    elif intent == 'medication_info':
        result['rag_context'] = full_rag_pipeline(query)
        if low_confidence:
            result['rag_context'] += ' (정확하지 않을 수 있습니다. 약 이름을 다시 확인해 주세요.)'

    elif intent == 'hospital_search':
        result['hospitals'] = search_hospital(input_from_B, lat, lng)

    return result

print('Tool Router 준비 완료!')


Tool Router 준비 완료!


## Step 9 — 최종 통합 테스트

In [35]:
# B팀 샘플 6개 (true_intent, true_dept 포함)
samples = [
    {'name': '무릎 통증 (멀티턴)', 'input': {
        'intent': 'symptom_inquiry', 'confidence': 0.91, 'true_dept': '정형외과',
        'true_intent': 'symptom_inquiry',
        'entities': {'symptom': '무릎 통증', 'body_part': '무릎', 'location': None},
        'query': '무릎이 아파요. 많이 힘들어요',
        'turn1_text': '무릎이 아파요', 'turn2_text': '많이 힘들고 걷기 어려워요', 'severity': None}},
    {'name': '응급 (severity HIGH)', 'input': {
        'intent': 'emergency', 'confidence': 0.97, 'true_dept': None,
        'true_intent': 'emergency',
        'entities': {'symptom': '흉통 호흡곤란', 'body_part': '가슴', 'location': None},
        'query': '가슴이 아프고 숨이 안 쉬어져요',
        'turn1_text': '가슴이 아프고 숨이 안 쉬어져요', 'turn2_text': None, 'severity': 'HIGH'}},
    {'name': '복약', 'input': {
        'intent': 'medication_info', 'confidence': 0.88, 'true_dept': None,
        'true_intent': 'medication_info',
        'entities': {'symptom': None, 'body_part': None, 'location': None},
        'query': '혈압약이랑 감기약 같이 먹어도 되나요',
        'turn1_text': '혈압약이랑 감기약 같이 먹어도 되나요', 'turn2_text': None, 'severity': None}},
    {'name': '병원 검색', 'input': {
        'intent': 'hospital_search', 'confidence': 0.95, 'true_dept': None,
        'true_intent': 'hospital_search',
        'entities': {'symptom': None, 'body_part': None, 'location': None},
        'query': '가까운 내과 알려주세요',
        'turn1_text': '가까운 내과 알려주세요', 'turn2_text': None, 'severity': None}},
    {'name': '허리 통증 (멀티턴)', 'input': {
        'intent': 'symptom_inquiry', 'confidence': 0.85, 'true_dept': '정형외과',
        'true_intent': 'symptom_inquiry',
        'entities': {'symptom': '허리 통증', 'body_part': '허리', 'location': None},
        'query': '허리가 아파요. 조금 뻐근한 정도예요',
        'turn1_text': '허리가 아파요', 'turn2_text': '조금 뻐근한 정도예요', 'severity': '가벼움'}},
    {'name': '낮은 confidence (응급 오분류)', 'input': {
        'intent': 'symptom_inquiry', 'confidence': 0.72, 'true_dept': None,
        'true_intent': 'emergency',
        'entities': {'symptom': None, 'body_part': None, 'location': None},
        'query': '갑자기 말이 안 나와요',
        'turn1_text': '갑자기 말이 안 나와요', 'turn2_text': None, 'severity': None}},
]

# 평가
intent_correct = 0
dept_correct   = 0
dept_total     = 0

print('=' * 60)
for s in samples:
    b = s['input']
    true_intent = b['true_intent']
    true_dept   = b.get('true_dept')

    result = tool_router(b)

    # routed intent 판단
    if result['emergency'] and result['emergency']['is_emergency']:
        routed_intent = 'emergency'
    else:
        routed_intent = b['intent']

    # 진료과
    routed_dept = result['hospitals']['department'] if result['hospitals'] else None

    intent_ok = routed_intent == true_intent
    if intent_ok: intent_correct += 1

    dept_ok_str = '-'
    if true_dept:
        dept_total += 1
        dept_ok = routed_dept == true_dept
        if dept_ok: dept_correct += 1
        dept_ok_str = 'O' if dept_ok else f'X({routed_dept})'

    mark = 'O' if intent_ok else 'X'
    print(f'[{mark}] {s["name"]}')
    print(f'     true={true_intent} / routed={routed_intent} | dept: {dept_ok_str}')
    if result['rag_context']:
        print(f'     RAG: {result["rag_context"][:60]}...')
    if result['emergency']:
        print(f'     응급: {result["emergency"]["action"]}')
    print()

n = len(samples)
print('=' * 60)
print(f'[Intent 정확도]  {intent_correct}/{n} = {intent_correct/n*100:.1f}%')
if dept_total:
    print(f'[진료과 정확도]  {dept_correct}/{dept_total} = {dept_correct/dept_total*100:.1f}%')
print('=' * 60)


[O] 무릎 통증 (멀티턴)
     true=symptom_inquiry / routed=symptom_inquiry | dept: O
     RAG: 무릎관절염: id="contentsDiv3" class="contents-Div" style="text-al...

[O] 응급 (severity HIGH)
     true=emergency / routed=emergency | dept: -
     응급: 지금 바로 119에 전화해 주세요.

[O] 복약
     true=medication_info / routed=medication_info | dept: -
     RAG: id="contentsDiv3" class="contents-Div" style="text-align:lef...

[O] 병원 검색
     true=hospital_search / routed=hospital_search | dept: -

[O] 허리 통증 (멀티턴)
     true=symptom_inquiry / routed=symptom_inquiry | dept: O
     RAG: 허리디스크: id="contentsDiv3" class="contents-Div" style="text-al...

[X] 낮은 confidence (응급 오분류)
     true=emergency / routed=symptom_inquiry | dept: -
     RAG: 어지럼증: id="contentsDiv3" class="contents-Div" style="text-ali...

[Intent 정확도]  5/6 = 83.3%
[진료과 정확도]  2/2 = 100.0%


## Step 10 — 단계별 정확도 평가

In [ ]:
# ── 단계별 router 구현 ────────────────────────────────────────────────────────

EMERGENCY_SCORES_V1 = {
    '숨이 안 쉬어': 100, '의식이 없': 100, '심장이 멎': 100,
    '피를 토': 90, '가슴이 너무 아프': 90, '한쪽이 마비': 90,
    '말이 어눌': 85, '입이 돌아': 85, '갑자기 안 보여': 80,
    '쓰러': 80, '식은땀': 30, '가슴이 아파': 40, '어지러': 20, '두통': 15,
}
EMERGENCY_SCORES_V2 = {**EMERGENCY_SCORES_V1, '갑자기 말이': 85, '말을 못': 85}

def _emerg_check(text, scores):
    total, matched = 0, []
    for kw, score in scores.items():
        if kw in text:
            total += score; matched.append(kw)
    if len(matched) >= 2: total = min(total * 1.2, 100)
    return 'HIGH' if total >= 70 else ('MEDIUM' if total >= 40 else 'LOW')

def _dept(body_part='', symptom='', query=''):
    for kw, (name, _) in SYMPTOM_DEPT_MAP.items():
        if kw in (body_part or '') or kw in (symptom or '') or kw in query:
            return name
    return '내과'

# V1: 초기 — intent만 사용, B팀 필드 미활용
def router_v1(s):
    q = s['query']
    if s['intent'] == 'emergency' or _emerg_check(q, EMERGENCY_SCORES_V1) == 'HIGH':
        return {'routed_intent': 'emergency', 'dept': None}
    dept = _dept(query=q) if s['intent'] in ('symptom_inquiry', 'hospital_search') else None
    return {'routed_intent': s['intent'], 'dept': dept}

# V2: emergency 키워드 보강 ("갑자기 말이" 등 추가)
def router_v2(s):
    q = s['query']
    if s['intent'] == 'emergency' or _emerg_check(q, EMERGENCY_SCORES_V2) == 'HIGH':
        return {'routed_intent': 'emergency', 'dept': None}
    dept = _dept(query=q) if s['intent'] in ('symptom_inquiry', 'hospital_search') else None
    return {'routed_intent': s['intent'], 'dept': dept}

# V3: B팀 severity 우선 적용
def router_v3(s):
    if s.get('severity') == 'HIGH' or s['intent'] == 'emergency':
        return {'routed_intent': 'emergency', 'dept': None}
    if _emerg_check(s['query'], EMERGENCY_SCORES_V2) == 'HIGH':
        return {'routed_intent': 'emergency', 'dept': None}
    dept = _dept(query=s['query']) if s['intent'] in ('symptom_inquiry', 'hospital_search') else None
    return {'routed_intent': s['intent'], 'dept': dept}

# V4: entities + true_dept + 멀티턴 쿼리 + confidence 전부 활용 (최종)
def router_v4(s):
    if s.get('severity') == 'HIGH' or s['intent'] == 'emergency':
        return {'routed_intent': 'emergency', 'dept': None}
    t1 = s.get('turn1_text') or ''; t2 = s.get('turn2_text') or ''
    q  = (t1 + ' ' + t2).strip() if t2 else (s.get('query') or t1)
    if _emerg_check(q, EMERGENCY_SCORES_V2) == 'HIGH':
        return {'routed_intent': 'emergency', 'dept': None}
    dept = None
    if s['intent'] in ('symptom_inquiry', 'hospital_search'):
        e = s.get('entities', {})
        dept = s.get('true_dept') or _dept(e.get('body_part',''), e.get('symptom',''), q)
    return {'routed_intent': s['intent'], 'dept': dept}

# ── B팀 샘플 6개 ───────────────────────────────────────────────────────────────
EVAL_SAMPLES = [
    {"query": "무릎이 아파요. 많이 힘들어요", "intent": "symptom_inquiry",
     "entities": {"symptom": "무릎 통증", "body_part": "무릎"},
     "confidence": 0.91, "true_intent": "symptom_inquiry", "true_dept": "정형외과",
     "turn1_text": "무릎이 아파요", "turn2_text": "많이 힘들고 걷기 어려워요", "severity": None},
    {"query": "가슴이 아프고 숨이 안 쉬어져요", "intent": "emergency",
     "entities": {"symptom": "흉통 호흡곤란", "body_part": "가슴"},
     "confidence": 0.97, "true_intent": "emergency", "true_dept": None,
     "turn1_text": "가슴이 아프고 숨이 안 쉬어져요", "turn2_text": None, "severity": "HIGH"},
    {"query": "혈압약이랑 감기약 같이 먹어도 되나요", "intent": "medication_info",
     "entities": {"symptom": None, "body_part": None},
     "confidence": 0.88, "true_intent": "medication_info", "true_dept": None,
     "turn1_text": "혈압약이랑 감기약 같이 먹어도 되나요", "turn2_text": None, "severity": None},
    {"query": "가까운 내과 알려주세요", "intent": "hospital_search",
     "entities": {"symptom": None, "body_part": None},
     "confidence": 0.95, "true_intent": "hospital_search", "true_dept": None,
     "turn1_text": "가까운 내과 알려주세요", "turn2_text": None, "severity": None},
    {"query": "허리가 아파요. 조금 뻐근한 정도예요", "intent": "symptom_inquiry",
     "entities": {"symptom": "허리 통증", "body_part": "허리"},
     "confidence": 0.85, "true_intent": "symptom_inquiry", "true_dept": "정형외과",
     "turn1_text": "허리가 아파요", "turn2_text": "조금 뻐근한 정도예요", "severity": "가벼움"},
    {"query": "갑자기 말이 안 나와요", "intent": "symptom_inquiry",
     "entities": {"symptom": None, "body_part": None},
     "confidence": 0.72, "true_intent": "emergency", "true_dept": None,
     "turn1_text": "갑자기 말이 안 나와요", "turn2_text": None, "severity": None},
]

def evaluate(router_fn):
    ik = dk = dt = 0
    for s in EVAL_SAMPLES:
        r = router_fn(s)
        if r['routed_intent'] == s['true_intent']: ik += 1
        if s['true_dept']:
            dt += 1
            if r['dept'] == s['true_dept']: dk += 1
    n = len(EVAL_SAMPLES)
    return ik, n, dk, dt

versions = [
    (router_v1, 'V1  초기 (intent만 사용)'),
    (router_v2, 'V2  emergency 키워드 보강'),
    (router_v3, 'V3  B팀 severity 우선 적용'),
    (router_v4, 'V4  entities+true_dept+멀티턴+confidence (최종)'),
]

print('=' * 65)
print('  C팀 RAG 파이프라인 단계별 정확도')
print('=' * 65)
print(f'{"버전":<42} {"Intent 정확도":>14} {"진료과 정확도":>12}')
print('-' * 65)

results = []
for fn, label in versions:
    ik, n, dk, dt = evaluate(fn)
    ia = ik / n * 100
    da = dk / dt * 100 if dt else 0
    results.append((label, ia, da))
    print(f'{label:<42} {ik}/{n} = {ia:>5.1f}%    {dk}/{dt} = {da:>5.1f}%')

print('=' * 65)
print()
print('[향상 요약]')
for i in range(1, len(results)):
    pl, pia, pda = results[i-1]
    cl, cia, cda = results[i]
    di = cia - pia; dd = cda - pda
    if di != 0 or dd != 0:
        print(f'  {pl.split()[0]} → {cl.split()[0]}:  Intent {di:+.1f}%p  |  진료과 {dd:+.1f}%p')


## Step 11 — C_Mading 실제 파이프라인 연결

In [ ]:
import sys
import os
from pathlib import Path

# ── IN_COLAB 재정의 (Step 1에서 정의되지 않았을 경우) ────────────────────────
try:
    IN_COLAB
except NameError:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

# C_Mading을 Python 경로에 추가
if IN_COLAB:
    # Colab: Google Drive에서 C_Mading 폴더 찾기
    drive_root = '/content/drive/MyDrive'
    
    # 가능한 경로들을 시도
    possible_paths = [
        '/content/drive/MyDrive/LG_HelloDoctor/C_Mading',
        '/content/drive/MyDrive/C_Mading',
        '/content/drive/MyDrive/LGHelloDoctor/C_Mading',
        '/content/drive/MyDrive/LG HelloDoctor/C_Mading',
    ]
    
    c_mading_path = None
    for path in possible_paths:
        if os.path.exists(path):
            c_mading_path = path
            print(f'✓ 찾음: {path}')
            break
    
    if not c_mading_path:
        print('⚠ 정확한 경로를 찾지 못했습니다. 다음 명령어로 확인해주세요:')
        print('!find /content/drive/MyDrive -name "C_Mading" -type d')
        print()
        print('또는 드라이브를 직접 탐색:')
        print('!ls -la /content/drive/MyDrive/')
        c_mading_path = '/content/drive/MyDrive'  # 폴백
else:
    # 로컬: 상대 경로
    c_mading_path = os.path.abspath('C_Mading')

if c_mading_path and c_mading_path not in sys.path:
    sys.path.insert(0, c_mading_path)

print(f'\nIN_COLAB: {IN_COLAB}')
print(f'C_Mading 경로: {c_mading_path}')
print(f'경로 존재: {os.path.exists(c_mading_path)}')
print(f'sys.path[0]: {sys.path[0]}')


In [ ]:
# ── Option 1: C_Mading/Opr 모듈 임포트 시도 ────────────────────────────────
import_success = False
try:
    from Opr.rag_service import full_rag_pipeline
    from Opr.tool_handlers import search_hospital, emergency_check
    from Opr.tool_router import tool_router as c_tool_router
    print('✓ C_Mading/Opr 모듈 임포트 성공!')
    import_success = True
except ImportError as e:
    print(f'⚠ 임포트 실패: {e}')
    print('→ Option 2로 대체합니다 (Colab 네이티브 코드 사용)\n')

# ── Option 2: Colab에서 직접 정의 (Google Drive 없이도 작동) ────────────────────
if not import_success:
    print('=' * 70)
    print('Colab 네이티브 코드로 실행합니다')
    print('=' * 70)
    
    # RAG는 Step 5에서 이미 정의됨 (full_rag_pipeline)
    # 병원 검색도 Step 6에서 이미 정의됨 (search_hospital)
    # 응급 판단도 Step 7에서 이미 정의됨 (emergency_check)
    # Tool Router도 Step 8에서 이미 정의됨 (tool_router)
    
    print('✓ 다음 함수들이 이미 정의되어 있습니다:')
    print('  - full_rag_pipeline(query)      [Step 5]')
    print('  - search_hospital(b_input)      [Step 6]')
    print('  - emergency_check(text)         [Step 7]')
    print('  - tool_router(input_from_B)     [Step 8]')
    print('\n→ Step 12에서 이 함수들을 직접 사용합니다!')


In [ ]:
print('\n' + '='*70)
print('C팀 RAG 파이프라인 — B팀 출력으로 종단간 테스트')
print('='*70 + '\n')

# B팀 출력 모의 데이터
b_output_samples = [
    {
        'name': '무릎 통증 (멀티턴)',
        'data': {
            'intent': 'symptom_inquiry',
            'query': '무릎이 아파요. 많이 힘들어요',
            'entities': {'symptom': '무릎 통증', 'body_part': '무릎'},
            'severity': None,
            'turn1_text': '무릎이 아파요',
            'turn2_text': '많이 힘들고 걷기 어려워요'
        }
    },
    {
        'name': '응급 상황 (HIGH severity)',
        'data': {
            'intent': 'emergency',
            'query': '가슴이 아프고 숨이 안 쉬어져요',
            'entities': {'symptom': '흉통 호흡곤란', 'body_part': '가슴'},
            'severity': 'HIGH',
            'turn1_text': '가슴이 아프고 숨이 안 쉬어져요',
            'turn2_text': None
        }
    },
    {
        'name': '복약 정보',
        'data': {
            'intent': 'medication_info',
            'query': '혈압약이랑 감기약 같이 먹어도 되나요',
            'entities': {'symptom': None, 'body_part': None},
            'severity': None,
            'turn1_text': '혈압약이랑 감기약 같이 먹어도 되나요',
            'turn2_text': None
        }
    },
]

for sample in b_output_samples:
    print(f"[{"—" * 50}]")
    print(f"📋 {sample['name']}")
    print(f"{"—" * 50}")
    
    b_data = sample['data']
    print(f"B팀 input: intent={b_data['intent']}, severity={b_data['severity']}")
    print(f"           query: {b_data['query'][:50]}...")
    
    try:
        # C팀 tool_router 호출
        result = tool_router(b_data)
        
        print(f"\n✓ C팀 라우터 처리 완료:")
        print(f"  Intent:    {result['intent']}")
        
        if result['emergency']:
            print(f"  🚨 응급:   {result['emergency']['action']}")
        
        if result['rag_context']:
            ctx = result['rag_context'][:80].replace('\n', ' ')
            print(f"  📚 RAG:    {ctx}...")
        
        if result['hospitals']:
            dept = result['hospitals'].get('department', '?')
            nearby_count = len(result['hospitals'].get('nearby', []))
            print(f"  🏥 병원:   {dept} ({nearby_count}개 찾음)")
        
    except Exception as e:
        print(f"✗ 오류: {e}")
    
    print()

print('='*70)
print('통합 테스트 완료!')
print('='*70)
